In [117]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Input
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [118]:
# Parameters
input_shape = (128, 128, 3)
batch_size = 32  
epochs = 10


In [119]:
data_dir = 'C:/Users/5A_Traders/Downloads/FYP_IntelliTrain/ImageClassification/datasets/archive/Chessman-image-dataset/Chess/'

In [120]:
# Data Augmentation with a validation split
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2  # Reserve 20% of the data for validation
)

In [121]:

train_generator = datagen.flow_from_directory(
    data_dir,
    target_size=input_shape[:2],
    batch_size=batch_size,
    class_mode='categorical',
    subset='training',  # set as training data
    shuffle=True
)

Found 118 images belonging to 6 classes.


In [122]:
validation_generator = datagen.flow_from_directory(
    data_dir,
    target_size=input_shape[:2],
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',  # set as validation data
    shuffle=True
)

Found 26 images belonging to 6 classes.


In [123]:
# Dynamically determine the number of classes from the training generator
num_classes = len(train_generator.class_indices)
print(f"Detected {num_classes} classes.")


Detected 6 classes.


In [124]:
base_model = tf.keras.applications.EfficientNetB0(
    input_shape=input_shape,
    include_top=False,
    weights='imagenet'
)

In [125]:
# Freeze the base model to keep its pretrained weights.
base_model.trainable = False

In [126]:
# Build a new model on top of the base model.
inputs = Input(shape=input_shape)
x = base_model(inputs, training=False)  # Set training=False to avoid updating BatchNorm layers
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)  # Add dropout for regularization
outputs = Dense(num_classes, activation='softmax')(x)
model = Model(inputs, outputs)

In [127]:
# --- Compile the Model ---
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)


In [128]:
# --- Callbacks for Optimized Training ---
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, verbose=1)
]

In [129]:
# --- Train the Model ---
history = model.fit(
    train_generator,
    epochs=epochs,
    validation_data=validation_generator,
    callbacks=callbacks
)

Epoch 1/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 37s 3s/step - accuracy: 0.2133 - loss: 1.7606 - val_accuracy: 0.2308 - val_loss: 1.7191 - learning_rate: 3.0000e-04
Epoch 2/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 7s 768ms/step - accuracy: 0.2307 - loss: 1.7360 - val_accuracy: 0.3077 - val_loss: 1.6907 - learning_rate: 3.0000e-04
Epoch 3/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 7s 691ms/step - accuracy: 0.2465 - loss: 1.6984 - val_accuracy: 0.3077 - val_loss: 1.6693 - learning_rate: 3.0000e-04
Epoch 4/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 7s 703ms/step - accuracy: 0.2471 - loss: 1.7038 - val_accuracy: 0.3077 - val_loss: 1.6525 - learning_rate: 3.0000e-04
Epoch 5/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 7s 725ms/step - accuracy: 0.2916 - loss: 1.6762 - val_accuracy: 0.3077 - val_loss: 1.6417 - learning_rate: 3.0000e-04
Epoch 6/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 6s 722ms/step - accuracy: 0.2714 - loss: 1.6573 - val_accuracy: 0.3077 - val_loss: 1.6354 - learning_rate: 3.0000e-04
Epoch 7/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 7s 710ms/step - accuracy: 0.3159 - loss: 1